# 1. Data Collection & Preprocessing

This notebook handles:
- Loading MODIS/VIIRS satellite fire detection data
- Filtering for California region (lat 32-42, lon -125 to -114)
- Creating a 0.25° spatial grid
- Labeling fire ignition events
- Monthly ignition distribution analysis

**Data Source:** [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/)

## Setup & Data Loading

> **Note:** Update the `fire_folder` path below to point to your local fire dataset directory.
> The original project used Google Drive via Colab.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Path to fire datasets folder
fire_folder = "/content/drive/MyDrive/Wildfire_Project/fire_datasets"

# Get all fire CSV files
fire_files = glob.glob(os.path.join(fire_folder, "*.csv"))

print("Fire files found:", fire_files)

# Read and combine
fire_df_list = [pd.read_csv(f) for f in fire_files]
fires = pd.concat(fire_df_list, ignore_index=True)

print("Total fire rows:", len(fires))

In [ ]:
ca_fires = fires[
    (fires["latitude"] >= 32) & (fires["latitude"] <= 42) &
    (fires["longitude"] >= -125) & (fires["longitude"] <= -114)
].copy()

print("California rows (before filtering):", len(ca_fires))

In [ ]:
ca_fires = ca_fires[
    (ca_fires["type"] == 0) &
    (ca_fires["confidence"].isin(["n", "h"]))
].copy()

print("California rows (after filtering):", len(ca_fires))

In [ ]:
ca_fires["acq_date"] = pd.to_datetime(ca_fires["acq_date"])

## Spatial Grid Creation

In [ ]:
lat_min, lat_max = 32, 42
lon_min, lon_max = -125, -114
grid_size = 0.25

ca_fires["lat_bin"] = ((ca_fires["latitude"] - lat_min) / grid_size).astype(int)
ca_fires["lon_bin"] = ((ca_fires["longitude"] - lon_min) / grid_size).astype(int)

print("Max lat_bin:", ca_fires["lat_bin"].max())
print("Max lon_bin:", ca_fires["lon_bin"].max())

In [ ]:
grid_day_fire = (
    ca_fires
    .groupby(["lat_bin", "lon_bin", "acq_date"])
    .size()
    .reset_index(name="fire_count")
)

grid_day_fire["fire_present"] = 1

print("Total grid-day fire rows:", len(grid_day_fire))

In [ ]:
date_range = pd.date_range(
    start="2019-01-01",
    end="2023-12-31",
    freq="D"
)

print("Total days:", len(date_range))
unique_cells = grid_day_fire[["lat_bin", "lon_bin"]].drop_duplicates()

print("Unique fire cells:", len(unique_cells))
print("Total days:", len(date_range))

In [ ]:
full_grid = (
    unique_cells.assign(key=1)
    .merge(pd.DataFrame({"acq_date": date_range, "key": 1}), on="key")
    .drop("key", axis=1)
)

print("Total rows in full grid:", len(full_grid))

In [ ]:
dataset = full_grid.merge(
    grid_day_fire[["lat_bin", "lon_bin", "acq_date", "fire_present"]],
    on=["lat_bin", "lon_bin", "acq_date"],
    how="left"
)

dataset["fire_present"] = dataset["fire_present"].fillna(0)

print("Total fire_present sum:", dataset["fire_present"].sum())

## Ignition Labeling

In [ ]:
dataset = dataset.sort_values(["lat_bin", "lon_bin", "acq_date"])

dataset["prev_7_fire"] = (
    dataset
    .groupby(["lat_bin", "lon_bin"])["fire_present"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).sum())
)

dataset["ignition"] = np.where(
    (dataset["fire_present"] == 1) &
    (dataset["prev_7_fire"] == 0),
    1,
    0
)

print("Total ignition events:", dataset["ignition"].sum())

In [ ]:
print("Type counts:")
print(ca_fires["type"].value_counts())

print("\nConfidence counts:")
print(ca_fires["confidence"].value_counts())

## Monthly Ignition Distribution

In [ ]:
# Extract month
dataset["month"] = dataset["acq_date"].dt.month

# Aggregate ignition per month
monthly_ignition = (
    dataset.groupby("month")["ignition"]
    .sum()
    .reset_index()
)

print(monthly_ignition)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(monthly_ignition["month"], monthly_ignition["ignition"])
plt.xlabel("Month")
plt.ylabel("Ignition Count")
plt.title("Monthly Ignition Distribution (California)")
plt.show()